# Where to Deploy Your LLM: A Full Hosting Landscape

A theory-only reference on where an LLM (with a focus on vLLM-based serving, though
most of this applies to TGI/SGLang too) can actually be hosted. Where you deploy
depends on three things: **project maturity, budget, and data-privacy requirements** —
not just raw performance. Hosting options fall into four broad categories, ranging from
fully managed serverless APIs down to local, fully offline servers. Reflects the
landscape as of mid-2026.

**Contents**
1. Serverless AI platforms (easiest & most cost-effective)
2. Managed enterprise AI hubs (best for cloud ecosystems)
3. Raw infrastructure / IaaS (best for complete control)
4. Kubernetes / self-managed at scale
5. Local hardware & on-premise (best for zero-cost & full privacy)
6. Quick reference: which to pick
7. Direct recommendation guide
8. Questions worth answering before you choose
9. Ecosystem context (vLLM version, TGI status)


## 1. Serverless AI Platforms (Easiest & Most Cost-Effective)

If you want to use frameworks like vLLM or TGI without managing virtual machines,
Kubernetes, or scaling infrastructure yourself, serverless container platforms are the
best starting point. You only pay for the compute time your model actually uses, and
most of these scale to zero when idle.

### RunPod
- Has a dedicated **vLLM Serverless worker**: point it at any Hugging Face model
  (Llama, Mistral, Qwen3, DeepSeek-R1, Phi-4, etc.), configure max model length and a
  few environment variables, and get a running OpenAI-compatible endpoint — no
  Dockerfile required.
- Deployment happens through the RunPod web console: create an endpoint, pick a GPU
  type, set worker count, and go live with a single click.
- Optional **FlashBoot** feature speeds up cold starts.
- Generally the **cheapest option for steady-state workloads** once the setup is
  stable.

### Vast.ai
- A marketplace for renting on-demand or interruptible GPUs directly from providers,
  often at prices well below the major clouds.
- More hands-on than RunPod's managed worker model — you're closer to raw compute
  rental — but extremely affordable for developers and startups comfortable with a
  bit more setup.

### Modal
- **Python-native**: decorate a function with its GPU requirement, `pip_install` vLLM
  inside the image definition, and `modal deploy` ships it as an autoscaling HTTPS
  endpoint.
- Handles container image builds, **snapshotting** (so cold starts can skip
  re-loading the model), persistent volumes, secrets management, and scheduled jobs.
- **Automatically scales down to zero** when nobody is using it — you write standard
  Python and it deploys vLLM container clusters instantly.
- Best developer experience if you're comfortable writing the vLLM loading/serving
  code yourself, rather than using a pre-packaged worker.

### Baseten
- More **production/enterprise-leaning**. Their **Truss** framework handles model
  packaging and supports both **TensorRT-LLM and vLLM** backends.
- Good fit for teams that need SLAs and monitoring dashboards out of the box.
- The only serverless platform here offering **self-hosted / hybrid deployment inside
  your own cloud**, with data-residency control at the enterprise tier.
- SOC 2 Type II certified and HIPAA compliant.

### Replicate
- A managed environment optimized specifically for running open-source models,
  providing out-of-the-box APIs for popular models (e.g. Llama 3, Mistral) with
  minimal configuration.
- A good fit when you want an API for a well-known open model without touching serving
  internals at all.

### Beam
- Similar niche to Modal and RunPod, known for clean artifact handling.
- Worth evaluating if the other options don't quite fit your workflow or pricing
  needs.


## 2. Managed Enterprise AI Hubs (Best for Cloud Ecosystems)

If your company already runs on a major cloud provider, its dedicated AI service hub is
usually the path of least resistance — these integrate directly with your existing
cloud storage, IAM/security permissions, and VPCs (Virtual Private Clouds), rather than
requiring a brand-new vendor relationship.

### Hugging Face Inference Endpoints
- Deploy any model directly from the Hugging Face Hub onto AWS or Azure with a few
  clicks.
- **Natively supports both TGI and vLLM** as serving backends.
- A strong option when you want production deployment with minimal setup time and
  don't need a fully custom container.

### AWS SageMaker / Google Vertex AI / Azure ML
- Enterprise-grade platforms that deploy models securely inside an **isolated cloud
  network**.
- Support "bring your own container," so vLLM can be packaged and run inside their
  managed inference endpoints.
- **Ideal when strict data-privacy compliance is required** (HIPAA, SOC 2), since
  these integrate with the compliance tooling your org likely already has in place.
- Heavier setup than the serverless platforms above — you own the container image and
  its integration with the platform's endpoint spec — but pays off when you're already
  deep in that ecosystem for logging, billing, and access control.


## 3. Raw Infrastructure / IaaS (Best for Complete Control)

If you want to manage the underlying operating system, build a custom Kubernetes
cluster, or heavily tune your serving engine's configuration, rent raw compute instead
of a managed platform.

### Specialized GPU Clouds
- **Lambda Labs, CoreWeave, FluidStack**, and similar providers offer highly reliable,
  enterprise-grade NVIDIA H100/A100 instances, often at a fraction of the cost of the
  legacy hyperscalers.
- A strong middle ground: dedicated GPU hardware without the general-purpose cloud
  overhead (or pricing) of AWS/GCP/Azure.

### Traditional Cloud VMs
- **AWS EC2** (e.g. `g5`, `p4` instances), **Google Compute Engine**, and **Microsoft
  Azure NV-series** VMs.
- You manually SSH into the server, install GPU drivers, and run your serving
  framework of choice yourself.
- Maximum flexibility and full control over the stack, at the cost of owning every
  layer of setup and maintenance.


## 4. Kubernetes / Self-Managed at Scale

A specific, common shape of "raw infrastructure" deployment worth calling out on its
own: running a purpose-built serving stack on top of a Kubernetes cluster, whether
that cluster lives on a specialized GPU cloud, a hyperscaler, or bare metal.

### vLLM Production Stack
- The **vLLM project's own reference Kubernetes deployment**: Helm charts, a request
  router, and autoscaling, maintained by the same team as vLLM itself.
- The 2026 roadmap includes: intelligent routing for agentic and multimodal
  workloads, automatic request migration if a vLLM instance fails, router-level safety
  policies/rate limits/logging, optional PII detection in the request path, and
  automated Helm/K8s/Docker image releases.
- The right choice when you want maximum control and are already operating your own
  GPU infrastructure.

### BentoML
- A general-purpose model-serving framework with vLLM runner support.
- Useful when you want more customization than the serverless platforms allow, but
  don't want to build the full Kubernetes stack from scratch yourself.


## 5. Local Hardware & On-Premise (Best for Zero-Cost & Full Privacy)

If you're developing locally, prototyping an application, or handling data so
sensitive it legally cannot leave your building, host it entirely offline.

### Your Laptop / Desktop
- **Ollama** or **llama.cpp** let you serve models directly from a MacBook or a
  Windows PC with an NVIDIA RTX card.
- The API is typically hosted at `http://localhost:11434` or a similar local port —
  no network egress at all.
- Zero ongoing cost, and the only option with **no data ever leaving the machine**.

### On-Premise Servers
- Companies buy their own physical servers populated with NVIDIA H100s or PCIe cards
  (e.g. the RTX 6000 Ada) to run private enterprise applications behind a corporate
  firewall.
- Makes sense when data sensitivity or regulatory requirements rule out any cloud
  option entirely, and the organization can justify the upfront hardware capital cost.


## 6. Quick Reference: Which to Pick

| You want | Go with |
|---|---|
| Fastest possible "model → endpoint" with minimal code | RunPod (vLLM worker) |
| Cheapest raw GPU rental, comfortable with more setup | Vast.ai |
| Full Python control, still low-ops, scales to zero | Modal |
| Enterprise SLAs, dashboards, maybe hybrid/on-prem | Baseten |
| An instant API for a well-known open model | Replicate |
| Deploy straight from the Hugging Face Hub | Hugging Face Inference Endpoints |
| Already committed to AWS/GCP/Azure, need compliance | SageMaker / Vertex AI / Azure ML |
| Dedicated GPU hardware without hyperscaler overhead | Lambda Labs / CoreWeave / FluidStack |
| Full manual control over the OS and stack | Traditional cloud VMs (EC2 / GCE / Azure NV) |
| Full control on your own GPU cluster | vLLM Production Stack (Kubernetes) |
| Zero cost, full privacy, just testing | Your own laptop (Ollama / llama.cpp) |
| Data can never leave the building | On-premise servers |


## 7. Direct Recommendation Guide

- **Choose RunPod or Modal** if you're an independent developer or startup looking for
  the cheapest way to host a vLLM-backed API.
- **Choose Hugging Face Inference Endpoints** if you want to deploy a model securely
  in production with minimal setup time.
- **Choose AWS SageMaker or Vertex AI** if you're an enterprise team with an existing
  corporate cloud architecture and compliance requirements.
- **Choose your local machine** if you're just testing code and want a completely
  free playground.


## 8. Questions Worth Answering Before You Choose

The "right" platform genuinely depends on context that only you have. Before
committing, it's worth being explicit about:

- Is this for a **personal/hobby project, a startup, or an established enterprise**?
- Is there a **target cloud provider you're required to use** (e.g. AWS or Azure for
  existing infra/compliance reasons)?
- What is the **approximate monthly budget** for hosting?
- Does the data involved have **privacy or regulatory constraints** that rule out
  shared/serverless infrastructure entirely?


## 9. Ecosystem Context (as of mid-2026)

- **vLLM v0.15.1** (February 2026) added PyTorch 2.10 support, along with RTX
  Blackwell (SM120) and H200 optimizations — the engine is under active development
  and gaining broader hardware support.
- **Hugging Face's TGI** (Text Generation Inference) entered **maintenance mode** and
  was **archived in March 2026**; Hugging Face now recommends **vLLM, SGLang,
  llama.cpp, or MLX** for new projects instead.
- This makes vLLM a comparatively safe long-term choice for production deployment —
  it isn't a framework at risk of being sunset, unlike TGI.
